# 10 · Modelo base

**Propósito:** entrenar y evaluar un primer modelo que estime, a T−2 h, la probabilidad de que un vuelo salga con
15 minutos o más de demora, y compararlo con una **referencia trivial**. Es la línea base técnica del proyecto: el
modelo definitivo de la segunda unidad tendrá que superarla.

- **Un solo modelo** aprende de los vuelos de Brasil y de EE. UU. (la variable `pais` le permite distinguirlos).
- **La evaluación se reporta por separado**: en el total, en EE. UU. y, sobre todo, en **LATAM Brasil**, que es el
  caso de negocio.
- **Partición temporal** (nunca al azar): train hasta `train_fin`, valid hasta `valid_fin` y test después.

**Entrada:** `gold/features_salida` (nunca `scale-test/`).

**Salida:** métricas en `docs/evidencias/modelo/` (JSON y CSV) y el modelo en `gold/modelos/modelo_base/`.

**Requisito:** notebook 08.

In [1]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

Destino: s3://latam-bigdata-lake-728fd64c
Periodo: 2023-01 a 2025-12


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import json
from datetime import UTC, datetime

import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import Imputer, OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-modelo-base")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/25 08:26:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/25 08:26:12 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


26/11/25 08:26:13 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


26/11/25 08:26:16 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to logs/spark-events/app-20261125082612-0007.inprogress. This is unsupported


In [3]:
EVID = RAIZ / "docs" / "evidencias" / "modelo"
EVID.mkdir(parents=True, exist_ok=True)
assert "scale-test" not in ruta("gold_features")

OBJETIVO = cfg["modelo"]["objetivo"]
NUMERICAS = list(cfg["features"]["numericas"])
CATEGORICAS = list(cfg["features"]["categoricas"])

gold = spark.read.parquet(ruta("gold_features"))
datos = gold.select(
    "id_vuelo", "particion", "pais", "es_latam", "origen", OBJETIVO,
    *[F.col(c).cast("double").alias(c) for c in NUMERICAS],
    *[F.coalesce(F.col(c).cast("string"), F.lit("desconocido")).alias(f"cat_{c}") for c in CATEGORICAS],
).where(F.col(OBJETIVO).isNotNull())
datos = datos.withColumn("pais_original", F.col("pais")).cache()

conteo = datos.groupBy("particion", "pais").agg(F.count("*").alias("vuelos"), F.avg(OBJETIVO).alias("tasa_demora"))
conteo.orderBy("particion", "pais").toPandas()

,particion,pais,vuelos,tasa_demora
0,test,BR,948446,0.179161
1,test,US,6817315,0.217451
2,train,BR,1337148,0.187842
3,train,US,10057865,0.208971
4,valid,BR,468230,0.206503
5,valid,US,3531898,0.196305


## 1. Referencia trivial

La referencia no aprende nada: para cada vuelo usa la **tasa histórica de demora de su aeropuerto de origen a esa
hora**, calculada solo con train (es la variable `tasa_hist_origen_hora` del notebook 08). Si no hay historia
suficiente, usa la tasa general de train. Cualquier modelo que no la supere no aporta.

In [4]:
train = datos.where(F.col("particion") == "train")
valid = datos.where(F.col("particion") == "valid")
test = datos.where(F.col("particion") == "test")
tasa_general = train.agg(F.avg(OBJETIVO)).first()[0]
print(f"Tasa general de demora en train: {tasa_general:.4f}")


def con_referencia(df):
    return df.withColumn("score_referencia", F.coalesce(F.col("tasa_hist_origen_hora"), F.lit(tasa_general)))

Tasa general de demora en train: 0.2065


## 2. Regresión logística

- Las variables categóricas (`pais`, `aerolinea`, categoría de vuelo del origen) se convierten a indicadores
  (*one-hot*). Una categoría que no apareció en train va a un grupo propio.
- Los valores faltantes de las numéricas se completan con la mediana de train (`Imputer`) y todas se estandarizan.
- Las clases están desbalanceadas (la mayoría de los vuelos sale a tiempo): cada vuelo demorado pesa
  `vuelos a tiempo / vuelos demorados` en el entrenamiento, para que el modelo no aprenda a decir siempre "a tiempo".
- Una variable numérica sin ningún dato en train no se puede imputar: se deja fuera y se informa.

In [5]:
con_dato = train.select([F.count(c).alias(c) for c in NUMERICAS]).first().asDict()
NUM_USADAS = [c for c in NUMERICAS if con_dato[c] > 0]
print("Variables numéricas sin datos en train (se omiten):", [c for c in NUMERICAS if c not in NUM_USADAS])

positivos = train.where(F.col(OBJETIVO) == 1).count()
negativos = train.where(F.col(OBJETIVO) == 0).count()
peso_positivo = negativos / max(positivos, 1)
train = train.withColumn("peso", F.when(F.col(OBJETIVO) == 1, F.lit(peso_positivo)).otherwise(F.lit(1.0)))
print(f"Train: {positivos:,} demorados y {negativos:,} a tiempo · peso de un demorado: {peso_positivo:.2f}")

cat_cols = [f"cat_{c}" for c in CATEGORICAS]
etapas = [
    StringIndexer(inputCols=cat_cols, outputCols=[f"{c}_idx" for c in cat_cols], handleInvalid="keep"),
    OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_cols], outputCols=[f"{c}_oh" for c in cat_cols],
                  handleInvalid="keep"),
    Imputer(inputCols=NUM_USADAS, outputCols=[f"{c}_imp" for c in NUM_USADAS], strategy="median"),
    VectorAssembler(inputCols=[f"{c}_imp" for c in NUM_USADAS] + [f"{c}_oh" for c in cat_cols],
                    outputCol="vector", handleInvalid="keep"),
    StandardScaler(inputCol="vector", outputCol="vector_esc", withMean=False, withStd=True),
    LogisticRegression(featuresCol="vector_esc", labelCol=OBJETIVO, weightCol="peso",
                       maxIter=cfg["modelo"]["max_iter"], regParam=cfg["modelo"]["reg_param"]),
]
inicio = datetime.now(UTC)
modelo = Pipeline(stages=etapas).fit(train)
print(f"Entrenado en {(datetime.now(UTC) - inicio).total_seconds():.0f} s")
modelo.write().overwrite().save(f"{BASE}/{tablas['gold_modelos']}/modelo_base")

Variables numéricas sin datos en train (se omiten): []


Train: 2,352,977 demorados y 9,042,036 a tiempo · peso de un demorado: 3.84


Entrenado en 183 s


## 3. Umbral de decisión (elegido en valid)

El modelo entrega una probabilidad. Para convertirla en alerta sí/no hace falta un umbral, que se elige en
**valid** (nunca en test) como el que maximiza F1, el equilibrio entre precisión y cobertura. El umbral por costo
operativo (cuánto cuesta una falsa alarma frente a una demora no avisada) queda para la segunda unidad.

In [6]:
def puntuar(df):
    return con_referencia(modelo.transform(df)).withColumn("score_modelo", vector_to_array("probability")[1])


def mejor_umbral(df, score):
    """Umbral que maximiza F1, con las probabilidades agrupadas en 200 tramos (una sola pasada)."""
    tramos = (
        df.withColumn("tramo", F.least(F.floor(F.col(score) * 200), F.lit(199)))
        .groupBy("tramo").agg(F.sum(OBJETIVO).alias("pos"), F.count("*").alias("n")).toPandas()
        .set_index("tramo").reindex(range(200), fill_value=0).sort_index(ascending=False)
    )
    vp = tramos.pos.cumsum()
    alertas = tramos.n.cumsum()
    precision = vp / alertas.where(alertas > 0)
    recall = vp / max(tramos.pos.sum(), 1)
    f1 = (2 * precision * recall / (precision + recall)).fillna(0)
    mejor = f1.idxmax()
    return mejor / 200


valid_p = puntuar(valid).cache()
UMBRALES = {"modelo": mejor_umbral(valid_p, "score_modelo"), "referencia": mejor_umbral(valid_p, "score_referencia")}
UMBRALES

{'modelo': 0.48, 'referencia': 0.185}

## 4. Evaluación en test

- **AUC-ROC:** qué tan bien ordena el modelo los vuelos de más a menos riesgo (0,5 = azar, 1 = perfecto).
- **AUC-PR:** lo mismo, pero enfocado en los vuelos demorados, que son minoría. Su piso es la tasa de demora del
  segmento.
- **Precisión:** de cada 100 alertas, cuántas fueron demoras reales.
- **Recall:** de cada 100 demoras, cuántas se avisaron.

In [7]:
test_p = puntuar(test).cache()
SEGMENTOS = {
    "Total": F.lit(True),
    "EE. UU.": F.col("pais_original") == "US",
    "Brasil (todas las aerolíneas)": F.col("pais_original") == "BR",
    "LATAM Brasil": F.col("es_latam"),
}


def metricas(df, score, umbral):
    n = df.count()
    if n == 0 or df.select(OBJETIVO).distinct().count() < 2:
        return {"vuelos": n}
    auc_roc = BinaryClassificationEvaluator(rawPredictionCol=score, labelCol=OBJETIVO,
                                            metricName="areaUnderROC").evaluate(df)
    auc_pr = BinaryClassificationEvaluator(rawPredictionCol=score, labelCol=OBJETIVO,
                                           metricName="areaUnderPR").evaluate(df)
    c = df.select(
        F.sum(((F.col(score) >= umbral) & (F.col(OBJETIVO) == 1)).cast("int")).alias("vp"),
        F.sum(((F.col(score) >= umbral) & (F.col(OBJETIVO) == 0)).cast("int")).alias("fp"),
        F.sum(((F.col(score) < umbral) & (F.col(OBJETIVO) == 1)).cast("int")).alias("fn"),
        F.avg(OBJETIVO).alias("tasa"),
    ).first()
    precision = c.vp / (c.vp + c.fp) if (c.vp + c.fp) else 0.0
    recall = c.vp / (c.vp + c.fn) if (c.vp + c.fn) else 0.0
    return {
        "vuelos": n, "tasa_demora": round(c.tasa, 4), "auc_roc": round(auc_roc, 4), "auc_pr": round(auc_pr, 4),
        "precision": round(precision, 4), "recall": round(recall, 4),
        "f1": round(2 * precision * recall / (precision + recall), 4) if precision + recall else 0.0,
        "pct_alertado": round((c.vp + c.fp) / n, 4), "umbral": round(umbral, 3),
    }


filas = []
for nombre, condicion in SEGMENTOS.items():
    parte = test_p.where(condicion)
    for modelo_nombre, score, clave_umbral in [("referencia", "score_referencia", "referencia"),
                                               ("regresion_logistica", "score_modelo", "modelo")]:
        filas.append({"segmento": nombre, "modelo": modelo_nombre,
                      **metricas(parte, score, UMBRALES[clave_umbral])})
tabla = pd.DataFrame(filas)
tabla

,segmento,modelo,vuelos,tasa_demora,auc_roc,auc_pr,precision,recall,f1,pct_alertado,umbral
0,Total,referencia,7765761,0.2128,0.6537,0.3148,0.2800,0.7381,0.4059,0.5609,0.185
1,Total,regresion_logistica,7765761,0.2128,0.6905,0.3890,0.3313,0.6152,0.4307,0.3951,0.480
2,EE. UU.,referencia,6817315,0.2175,0.6543,0.3177,0.2862,0.7451,0.4136,0.5660,0.185
3,EE. UU.,regresion_logistica,6817315,0.2175,0.6914,0.3958,0.3351,0.6263,0.4366,0.4064,0.480
4,Brasil (todas las aerolíneas),referencia,948446,0.1792,0.6314,0.2749,0.2312,0.6767,0.3447,0.5243,0.185
5,Brasil (todas las aerolíneas),regresion_logistica,948446,0.1792,0.6755,0.3331,0.2963,0.5185,0.3771,0.3135,0.480
6,LATAM Brasil,referencia,298582,0.1896,0.6439,0.2714,0.2508,0.7026,0.3696,0.5313,0.185
7,LATAM Brasil,regresion_logistica,298582,0.1896,0.6786,0.3299,0.3227,0.4752,0.3843,0.2792,0.480


## 5. ¿Supera a la referencia?

In [8]:
comparacion = tabla.pivot(index="segmento", columns="modelo", values="auc_roc")
if {"referencia", "regresion_logistica"} <= set(comparacion.columns):
    comparacion["mejora_auc_roc"] = comparacion["regresion_logistica"] - comparacion["referencia"]
    pr = tabla.pivot(index="segmento", columns="modelo", values="auc_pr")
    comparacion["mejora_auc_pr_relativa"] = (pr["regresion_logistica"] / pr["referencia"] - 1).round(3)
comparacion

modelo,referencia,regresion_logistica,mejora_auc_roc,mejora_auc_pr_relativa
segmento,,,,
Brasil (todas las aerolíneas),0.6314,0.6755,0.0441,0.212
EE. UU.,0.6543,0.6914,0.0371,0.246
LATAM Brasil,0.6439,0.6786,0.0347,0.216
Total,0.6537,0.6905,0.0368,0.236


In [9]:
resultado = {
    "ejecucion_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    "objetivo": OBJETIVO,
    "periodo": [DESDE, HASTA],
    "particion_temporal": cfg["particion_temporal"],
    "variables_numericas": NUM_USADAS,
    "variables_categoricas": CATEGORICAS,
    "peso_positivo": round(peso_positivo, 3),
    "umbrales_elegidos_en_valid": UMBRALES,
    "vuelos": {r["particion"] + "_" + r["pais"]: int(r["vuelos"]) for r in conteo.collect()},
    "test": filas,
}
(EVID / "metricas_modelo_base.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2),
                                                encoding="utf-8")
tabla.to_csv(EVID / "metricas_modelo_base.csv", index=False)
print("Métricas guardadas en docs/evidencias/modelo/")

Métricas guardadas en docs/evidencias/modelo/


## 6. Plan de mejora para la segunda unidad

Lo que se propone a partir de este resultado, cada punto con su razón:

1. **Modelos de árboles (Random Forest y GBT).** La regresión logística solo capta relaciones lineales; el efecto de
   la red y del clima probablemente es no lineal y con interacciones (por ejemplo, tormenta *y* hub congestionado).
2. **Umbral por costo operativo.** Hoy el umbral maximiza F1. Para el CCO una demora no avisada y una falsa alarma
   no cuestan lo mismo: el umbral debe minimizar el costo esperado con una razón de costos explícita.
3. **Capacidad del turno.** Medir precisión y recall cuando el CCO solo puede revisar un número fijo de vuelos por
   aeropuerto y día.
4. **Más señal de propagación.** Agregar las llegadas atrasadas al aeropuerto de origen antes de T−2 h, que es lo más
   cercano a la rotación del avión disponible en datos públicos.
5. **Calibración y estabilidad por segmento.** Revisar que la probabilidad sea confiable por país, aerolínea y
   aeropuerto, y vigilar la deriva entre periodos.
6. **Objetivo de 60 minutos.** Entrenar también la alerta roja, que es la de mayor costo para la operación.

In [10]:
spark.stop()